# 00 – Transform Firewall CSV Exports → Parquet

This notebook supports **two** CSV export formats via the `FORMAT` variable:

| `FORMAT` | Parser | Description |
|---|---|---|
| `"export"` | `FirewallExportParser` | Format `;`-séparé, sans en-tête |
| `"kibana"` | `KibanaExportParser` | Export Kibana/Elasticsearch (`,` séparé, avec en-tête) |

### Format `"export"` (pas d'en-tête, séparateur `;`)
```
2025-03-20 01:29:24;94.102.61.47;159.84.146.99;TCP;52502;3178;999;DENY;eth0;;6
```
Colonnes : `timestamp ; src_ip ; dst_ip ; proto ; src_port ; dst_port ; rule ; action ; interface_in ; interface_out ; fw`

### Format `"kibana"` (export Kibana/Elasticsearch)
```
@timestamp,_id,_index,...,action,action.keyword,dstport,...,ipsrc,...
"Mar 2, 2026 @ 20:45:01.000",...
```

In [22]:
import sys
from pathlib import Path

# Make the project root importable when running from the notebooks/ folder
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\cyraptor\Documents\PROJECTS\Challenge-CyberSecu-SISE


## 1 – Configuration des chemins

Renseignez ici :
- **`FORMAT`** : `"export"` pour le format `;`-séparé, `"kibana"` pour l'export Kibana/ES
- **`CSV_PATH`** : chemin vers le fichier CSV source
- **`PARQUET_PATH`** : chemin du fichier Parquet de sortie
- Filtre de dates et suppression de `fw` via les variables ci-dessous.

In [23]:
from pathlib import Path

# ── À MODIFIER ────────────────────────────────────────────────────────────────
FORMAT       = "kibana"   # "export" = format ;-séparé  |  "kibana" = export Kibana/ES
CSV_PATH     = "C:/Users/cyraptor/Documents/PROJECTS/Challenge-CyberSecu-SISE/data/raw/logs.csv"
PARQUET_PATH = PROJECT_ROOT / "data" / "processed" / "firewall_logs.parquet"

# Filtre de dates (optionnel) — mettre FILTER_DATES = False pour désactiver
FILTER_DATES = False
DATE_FROM    = "2025-11-01"   # inclus
DATE_TO      = "2026-02-28"   # inclus
# ──────────────────────────────────────────────────────────────────────────────

if FORMAT not in ("export", "kibana"):
    raise ValueError(f"FORMAT doit être 'export' ou 'kibana', reçu : {FORMAT!r}")
if not Path(CSV_PATH).exists():
    raise FileNotFoundError(f"Fichier introuvable : {CSV_PATH}")

print(f"Format  : {FORMAT}")
print(f"Source  : {CSV_PATH}")
print(f"Sortie  : {PARQUET_PATH}")
if FILTER_DATES:
    print(f"Filtre  : {DATE_FROM}  →  {DATE_TO}")

Format  : kibana
Source  : C:/Users/cyraptor/Documents/PROJECTS/Challenge-CyberSecu-SISE/data/raw/logs.csv
Sortie  : C:\Users\cyraptor\Documents\PROJECTS\Challenge-CyberSecu-SISE\data\processed\firewall_logs.parquet


## 2 – Parser le fichier CSV

Le parser est sélectionné automatiquement selon `FORMAT`.  
Utilisable aussi directement dans l'application :

```python
from parsers import ParserFactory

# Format ;-séparé (sans en-tête)
parser = ParserFactory.create("firewall_export")

# Export Kibana/Elasticsearch
parser = ParserFactory.create("kibana_export")

df = parser.parse("/chemin/vers/fichier.csv")
```

In [24]:
from parsers import FirewallExportParser, KibanaExportParser

if FORMAT == "kibana":
    parser = KibanaExportParser()
else:
    parser = FirewallExportParser()

df = parser.parse(str(CSV_PATH))

is_valid, errors = parser.validate(df)
print(f"Parser  : {type(parser).__name__}")
print(f"Valid   : {is_valid}  |  Errors: {errors or 'none'}")
print(f"\nShape   : {df.shape}")
print(f"Columns : {list(df.columns)}")
df

Parser  : KibanaExportParser
Valid   : True  |  Errors: none

Shape   : (22235, 18)
Columns : ['timestamp', 'hostname', 'action', 'fw', 'rule', 'interface_in', 'interface_out', 'src_ip', 'dst_ip', 'len', 'ttl', 'id', 'df', 'proto', 'src_port', 'dst_port', 'window', 'flags']


,timestamp,hostname,action,fw,rule,interface_in,interface_out,src_ip,dst_ip,len,ttl,id,df,proto,src_port,dst_port,window,flags
0,2026-03-03 11:19:20,<NA>,DENY,<NA>,34,eth0,<NA>,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,36334,<NA>,<NA>
1,2026-03-03 11:19:19,<NA>,DENY,<NA>,34,eth0,<NA>,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,36334,<NA>,<NA>
2,2026-03-03 11:19:18,<NA>,DENY,<NA>,34,eth0,<NA>,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,36334,<NA>,<NA>
3,2026-03-03 11:19:17,<NA>,DENY,<NA>,34,eth0,<NA>,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,36334,<NA>,<NA>
4,2026-03-03 11:19:16,<NA>,DENY,<NA>,34,eth0,<NA>,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,36334,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22230,2026-03-02 18:47:31,<NA>,DENY,<NA>,34,eth0,<NA>,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,37524,<NA>,<NA>
22231,2026-03-02 18:47:31,<NA>,DENY,<NA>,34,eth0,<NA>,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,37524,<NA>,<NA>
22232,2026-03-02 18:47:30,<NA>,DENY,<NA>,34,eth0,<NA>,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,37524,<NA>,<NA>
22233,2026-03-02 18:47:30,<NA>,DENY,<NA>,34,eth0,<NA>,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,37524,<NA>,<NA>


In [25]:
# Inspect dtypes and a quick summary
print(df.dtypes)
print()
df[["timestamp", "action", "src_ip", "dst_ip", "src_port", "dst_port", "rule"]].head(10)

timestamp        datetime64[ns]
hostname                 object
action                 category
fw                       object
rule                      Int64
interface_in             object
interface_out            object
src_ip                   object
dst_ip                   object
len                       Int64
ttl                       Int64
id                        Int64
df                         bool
proto                  category
src_port                  Int64
dst_port                  Int64
window                    Int64
flags                    object
dtype: object



,timestamp,action,src_ip,dst_ip,src_port,dst_port,rule
0,2026-03-03 11:19:20,DENY,172.17.0.2,172.17.0.3,514,36334,34
1,2026-03-03 11:19:19,DENY,172.17.0.2,172.17.0.3,514,36334,34
2,2026-03-03 11:19:18,DENY,172.17.0.2,172.17.0.3,514,36334,34
3,2026-03-03 11:19:17,DENY,172.17.0.2,172.17.0.3,514,36334,34
4,2026-03-03 11:19:16,DENY,172.17.0.2,172.17.0.3,514,36334,34
5,2026-03-03 11:19:15,DENY,172.17.0.2,172.17.0.3,514,36334,34
6,2026-03-03 11:19:14,DENY,172.17.0.2,172.17.0.3,514,36334,34
7,2026-03-03 11:19:13,DENY,172.17.0.2,172.17.0.3,514,36334,34
8,2026-03-03 11:19:12,DENY,172.17.0.2,172.17.0.3,514,36334,34
9,2026-03-03 11:19:11,DENY,172.17.0.2,172.17.0.3,514,36334,34


## 3 – Post-traitement : filtre de dates & suppression de `fw`

Conformément aux consignes :
- la colonne `fw` (numéro du firewall) est **supprimée**
- seules les lignes entre **novembre 2025** et **février 2026** sont conservées (si `FILTER_DATES = True`)

In [26]:
df_out = df.copy()

# ── Distribution des valeurs FW (informatif) ───────────────────────────────
print("Distribution des valeurs FW (toutes conservées) :")
print(df_out["fw"].value_counts().sort_index().to_string())
print()

# ── Suppression de la colonne fw ───────────────────────────────────────────
df_out = df_out.drop(columns=["fw", "interface_out"], errors="ignore")

# ── Filtre de dates (optionnel) ────────────────────────────────────────────
if FILTER_DATES:
    before = len(df_out)
    mask   = (df_out["timestamp"] >= DATE_FROM) & (df_out["timestamp"] <= DATE_TO)
    df_out = df_out.loc[mask].reset_index(drop=True)
    print(f"Filtre dates : {before:,} → {len(df_out):,} lignes  ({DATE_FROM} → {DATE_TO})")
else:
    print(f"Filtre de dates désactivé — {len(df_out):,} lignes conservées")

print(f"\nColonnes : {list(df_out.columns)}")
df_out.head(5)

Distribution des valeurs FW (toutes conservées) :
Series([], )

Filtre de dates désactivé — 22,235 lignes conservées

Colonnes : ['timestamp', 'hostname', 'action', 'rule', 'interface_in', 'src_ip', 'dst_ip', 'len', 'ttl', 'id', 'df', 'proto', 'src_port', 'dst_port', 'window', 'flags']


,timestamp,hostname,action,rule,interface_in,src_ip,dst_ip,len,ttl,id,df,proto,src_port,dst_port,window,flags
0,2026-03-03 11:19:20,<NA>,DENY,34,eth0,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,36334,<NA>,<NA>
1,2026-03-03 11:19:19,<NA>,DENY,34,eth0,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,36334,<NA>,<NA>
2,2026-03-03 11:19:18,<NA>,DENY,34,eth0,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,36334,<NA>,<NA>
3,2026-03-03 11:19:17,<NA>,DENY,34,eth0,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,36334,<NA>,<NA>
4,2026-03-03 11:19:16,<NA>,DENY,34,eth0,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,36334,<NA>,<NA>


## 4 – Sauvegarde en Parquet

`save_parquet` et `load_parquet` sont dans `utils.io` et peuvent être importés partout dans l'application.

In [27]:
from utils.io import save_parquet, load_parquet

saved = save_parquet(df_out, PARQUET_PATH, compression="snappy")
print(f"Sauvegardé → {saved}  ({saved.stat().st_size:,} bytes)")

Sauvegardé → C:\Users\cyraptor\Documents\PROJECTS\Challenge-CyberSecu-SISE\data\processed\firewall_logs.parquet  (194,178 bytes)


## 5 – Reload & vérification

In [28]:
df_loaded = load_parquet(PARQUET_PATH)

print(f"Rows loaded: {len(df_loaded)}")
print(f"Dtypes preserved: {dict(df_loaded.dtypes)}")
df_loaded

Rows loaded: 22235
Dtypes preserved: {'timestamp': dtype('<M8[ns]'), 'hostname': dtype('O'), 'action': CategoricalDtype(categories=['DENY', 'PERMIT'], ordered=False, categories_dtype=object), 'rule': Int64Dtype(), 'interface_in': dtype('O'), 'src_ip': dtype('O'), 'dst_ip': dtype('O'), 'len': Int64Dtype(), 'ttl': Int64Dtype(), 'id': Int64Dtype(), 'df': dtype('bool'), 'proto': CategoricalDtype(categories=['TCP'], ordered=False, categories_dtype=object), 'src_port': Int64Dtype(), 'dst_port': Int64Dtype(), 'window': Int64Dtype(), 'flags': dtype('O')}


,timestamp,hostname,action,rule,interface_in,src_ip,dst_ip,len,ttl,id,df,proto,src_port,dst_port,window,flags
0,2026-03-03 11:19:20,None,DENY,34,eth0,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,36334,<NA>,None
1,2026-03-03 11:19:19,None,DENY,34,eth0,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,36334,<NA>,None
2,2026-03-03 11:19:18,None,DENY,34,eth0,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,36334,<NA>,None
3,2026-03-03 11:19:17,None,DENY,34,eth0,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,36334,<NA>,None
4,2026-03-03 11:19:16,None,DENY,34,eth0,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,36334,<NA>,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22230,2026-03-02 18:47:31,None,DENY,34,eth0,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,37524,<NA>,None
22231,2026-03-02 18:47:31,None,DENY,34,eth0,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,37524,<NA>,None
22232,2026-03-02 18:47:30,None,DENY,34,eth0,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,37524,<NA>,None
22233,2026-03-02 18:47:30,None,DENY,34,eth0,172.17.0.2,172.17.0.3,<NA>,<NA>,<NA>,False,TCP,514,37524,<NA>,None
